# Module 1 — 70 ans d'IA en 30 minutes

**CONSEIL :** Commencez par enregistrer une copie de ce notebook dans votre Google Drive (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes !

**Objectifs d'apprentissage**
- Situer l'IA générative dans l'histoire de l'IA (symbolique → ML → DL → Transformers → LLM)
- Voir la **même tâche** (classer un texte par parti politique) traverser chaque époque
- Comprendre ce que chaque génération ajoute à la précédente
- Voir **un modèle apprendre** depuis des données, et lire ce qu'il a appris
- Apprivoiser un premier appel à un grand modèle de langage

## Running example : un peu de politique française

On va se balader dans 70 ans d'IA avec une question fil rouge : **« à quel parti appartient l'auteur de cette déclaration ? »**.

Le RN aime parler d'immigration. LFI préfère les salaires et le peuple. LR adore le marché libre et les entreprises. EELV milite pour la transition écologique et le climat. Vous voyez l'idée.

> En séance, nous remplacerons le mini-corpus ci-dessous par un vrai dataset de tweets de députés. Pour l'instant, 16 phrases suffisent à voir 70 ans d'IA défiler.

## Setup

In [ ]:
# Si vous êtes sur Colab, dé-commentez :
# !pip install -q scikit-learn datasets transformers anthropic

import pandas as pd

In [ ]:
# 16 déclarations courtes, 4 par parti
data = pd.DataFrame({
    "texte": [
        # RN
        "Il faut réduire l'immigration et défendre nos frontières.",
        "L'identité française doit être protégée.",
        "Sécurité, frontières, immigration : nos priorités sont claires.",
        "Notre identité et nos traditions face à la submersion migratoire.",
        # LFI
        "Augmentons les salaires et taxons les plus riches.",
        "Le peuple souffre pendant que la finance s'enrichit.",
        "Une révolution citoyenne contre l'oligarchie est nécessaire.",
        "Taxons les riches pour financer les services publics et les salaires.",
        # LR
        "Le marché libre et la baisse des charges relanceront l'économie.",
        "Soutenons les entreprises et libérons les énergies.",
        "Moins d'État, plus de responsabilité et de liberté économique.",
        "Baissons les charges des entreprises pour relancer le marché du travail.",
        # EELV
        "La transition écologique est urgente et juste.",
        "Le climat et la biodiversité au cœur des politiques publiques.",
        "Sortons des énergies fossiles, investissons dans le renouvelable.",
        "Une transition écologique juste pour préserver le climat.",
    ],
    "parti": ["RN", "RN", "RN", "RN",
              "LFI", "LFI", "LFI", "LFI",
              "LR", "LR", "LR", "LR",
              "EELV", "EELV", "EELV", "EELV"],
})
data

## 1.1 IA symbolique (années 50-80)

Nos ancêtres en IA n'avaient ni Internet, ni puissance de calcul, ni montagnes de données.
Leur arme : **des règles écrites à la main** par un expert humain.

Concrètement : pour reconnaître un texte du RN, on dresse une liste de mots qui reviennent souvent chez eux (« immigration », « frontière »…). Si un de ces mots apparaît, c'est du RN. Voilà notre première « IA ».

**Force** : c'est lisible, on sait exactement ce que le programme fait.
**Limite** : c'est très fragile. Une métaphore ou une formule détournée, et tout casse.

In [ ]:
# Une mini "IA" qui tient en 10 lignes : on cherche des mots-clés dans le texte
RULES = {
    "RN":   ["immigration", "frontière", "identité"],
    "LFI":  ["salaire", "riche", "peuple"],
    "LR":   ["marché", "libre", "charges", "entreprise"],
    "EELV": ["écolog", "transition", "climat"],
}

def predire_par_regles(texte):
    texte = texte.lower()
    for parti, mots in RULES.items():
        if any(m in texte for m in mots):
            return parti
    return "?"

data["pred_rules"] = data["texte"].apply(predire_par_regles)
data[["parti", "pred_rules"]].head(10)

### Hack Time

- Ajoutez un mot-clé par parti dans le dictionnaire `RULES`
- Comptez le nombre de prédictions correctes (`(data["parti"] == data["pred_rules"]).sum()`)
- Essayez d'écrire une phrase qui « casse » votre classifier (ironie, métaphore, mot d'un autre parti)

In [ ]:
# Votre code ici

## 1.2 Machine Learning statistique (années 90-2010)

L'idée centrale du ML : **arrêter d'écrire les règles à la main et les apprendre depuis les données**.

Reformulons : en 1.1 on écrivait nous-mêmes que *« immigration → RN »*. Le « poids » du mot-clé était fixé en dur (présent ou absent, point final).

En ML, on procède plutôt en trois temps :
1. On **prépare des features** simples (par exemple : « le mot 'immigration' est-il présent ? »)
2. On donne au modèle des **exemples étiquetés** (les 16 phrases ci-dessus)
3. On laisse le modèle découvrir tout seul **combien chaque mot pèse pour chaque parti**

Le modèle qu'on va utiliser est la **régression logistique** — un cousin proche de la régression linéaire que beaucoup d'entre vous ont déjà rencontrée. Avantage : à la fin, on peut **lire ses coefficients** comme on lirait les coefficients d'une régression OLS.

### 1.2.1 Préparer les features

In [ ]:
# Pour chaque mot-clé, on crée une colonne 0/1 : "ce mot apparaît-il dans le texte ?"
MOTS_CLES = ["immigration", "frontière", "identité",
             "salaire", "riche", "peuple",
             "marché", "libre", "charge", "entreprise",
             "écolog", "transition", "climat"]

def extraire_features(texte):
    texte = texte.lower()
    return {mot: int(mot in texte) for mot in MOTS_CLES}

features = pd.DataFrame([extraire_features(t) for t in data["texte"]])
features.insert(0, "parti", data["parti"])
features

Vous obtenez une **matrice de features** : chaque ligne = une phrase, chaque colonne = la présence (1) ou l'absence (0) d'un mot-clé.

C'est sur cette matrice que le modèle va travailler. Plus de texte brut, juste des 0 et des 1.

### 1.2.2 Train/test split — séparer ce qu'on montre du reste

In [ ]:
from sklearn.model_selection import train_test_split

X = pd.DataFrame([extraire_features(t) for t in data["texte"]])
y = data["parti"]

# On garde 4 phrases pour tester (1 par parti), 12 pour entraîner (3 par parti)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=4, stratify=y, random_state=42
)

print(f"Train : {len(X_train)} phrases / Test : {len(X_test)} phrases")

### 1.2.3 Voir le modèle apprendre

In [ ]:
from sklearn.linear_model import LogisticRegression

# Le modèle vide, qui ne sait encore rien
modele = LogisticRegression(max_iter=1000)

# Il apprend depuis les 12 phrases du train (et leur étiquette)
modele.fit(X_train, y_train)

# On lit ce qu'il a appris : ses coefficients
# Une ligne par parti, une colonne par mot-clé.
# Plus le coefficient est élevé, plus le mot tire vers ce parti.
coefs = pd.DataFrame(
    modele.coef_,
    columns=MOTS_CLES,
    index=modele.classes_,
).round(2)
coefs

**Lisez attentivement cette table.** Le modèle a découvert tout seul, sans qu'on lui dise rien :
- que `immigration` et `frontière` poussent vers RN,
- que `salaire` et `riche` poussent vers LFI,
- que `marché` et `entreprise` poussent vers LR,
- que `transition` et `climat` poussent vers EELV.

Bref, **il a réinventé nos règles de la section 1.1, mais en chiffres** — et avec des nuances en plus.

### Hack Time — jouez avec l'entraînement

- Ajoutez une 17ᵉ phrase à `data` (par exemple un message LFI sur l'écologie) et relancez tout ce qui précède. Les coefficients changent-ils beaucoup ?
- Que se passe-t-il si vous mettez `random_state=0` au lieu de `42` dans le split ? Le modèle apprend-il la même chose ?
- Retirez un mot-clé de `MOTS_CLES` et regardez : sur quel autre mot le modèle se rabat-il ?

### 1.2.4 Tester sur des phrases que le modèle n'a jamais vues

In [ ]:
# Précision sur train (déjà vu) vs test (jamais vu)
print(f"Précision train : {modele.score(X_train, y_train):.2f}")
print(f"Précision test  : {modele.score(X_test, y_test):.2f}")

# Détail des prédictions sur le test
preds = pd.DataFrame({
    "parti_vrai":    y_test.values,
    "parti_predit":  modele.predict(X_test),
})
preds

### Hack Time — jouez avec le test

Composez vous-même une phrase et faites-la classer par le modèle :

In [ ]:
ma_phrase = "Il faut taxer les multinationales pour financer l'hôpital public."

mes_features = pd.DataFrame([extraire_features(ma_phrase)])
prediction = modele.predict(mes_features)[0]
probabilites = pd.Series(modele.predict_proba(mes_features)[0], index=modele.classes_).round(2)

print(f"Prédiction : {prediction}")
print()
print("Probabilités :")
print(probabilites)

- Modifiez `ma_phrase` pour faire dire chaque parti au modèle
- Essayez une phrase **sans aucun mot-clé** de la liste. Que prédit-il ? Avec quelle confiance ?
- Essayez une phrase qui contient des mots-clés de **plusieurs** partis (un mélange RN + EELV par ex.). Comment le modèle tranche-t-il ?

## 1.3 Deep Learning (2010-2017)

Bascule majeure : on **arrête de fabriquer les features à la main**.

Au lieu de notre liste `MOTS_CLES` (qu'on avait choisie nous-mêmes), on laisse un réseau de neurones apprendre lui-même une représentation des textes (les fameux **embeddings**, qu'on verra en J3 text mining).

Pour notre tâche, c'est plus complexe à mettre en place mais ça commence à mieux marcher — surtout quand on a beaucoup de données.

(On ne va pas coder un réseau de neurones ce matin. Mais retenez le saut conceptuel : **représentation apprise**, plus de feature engineering manuel.)

> **Pour voir un réseau de neurones apprendre en direct** : ouvrez [playground.tensorflow.org](https://playground.tensorflow.org/) dans un nouvel onglet. Choisissez un dataset (en haut à gauche), cliquez sur Play, et regardez les couches « apprendre » à séparer les points en quelques secondes. C'est la même mécanique sous le capot des LLMs — en beaucoup plus gros.

## 1.4 Transformers (2017→)

**2017 : *Attention is all you need***. Une nouvelle architecture, le **transformer**, débarque chez Google.

Pourquoi ça change tout ? Parce que pour la première fois, un modèle peut être pré-entraîné sur d'énormes quantités de texte, **comprendre le sens des mots dans leur contexte**, et être ensuite adapté à n'importe quelle tâche avec très peu d'exemples.

L'innovation clé : le **mécanisme d'attention**. Chaque mot regarde tous les autres pour saisir son rôle exact dans la phrase. (Schéma en slide.)

### 1.4.1 Un test piégeux : 4 phrases sans nos mots-clés

Voici 4 phrases qui parlent des thèmes de nos partis… **mais sans utiliser aucun des mots-clés** qu'on a choisis en 1.2 (`immigration`, `salaire`, `marché`, `écolog`…). Synonymes uniquement.

In [ ]:
phrases_pieges = pd.DataFrame({
    "texte": [
        "L'arrivée massive d'étrangers doit être limitée.",        # thème RN, sans "immigration"
        "Augmentons la rémunération des plus modestes.",           # thème LFI, sans "salaire"
        "Encourageons les sociétés et l'esprit d'initiative.",     # thème LR, sans "entreprise"
        "Préservons notre planète et nos écosystèmes.",            # thème EELV, sans "écolog"
    ],
    "parti_vrai": ["RN", "LFI", "LR", "EELV"],
})
phrases_pieges

### 1.4.2 Quatre méthodes, de plus en plus puissantes

On va lancer 4 modèles sur ces mêmes phrases et regarder qui s'en sort :

| Méthode | Approche | Qui c'est ? |
|---|---|---|
| ① Règles | Mots-clés écrits à la main | Section 1.1 |
| ② Régression logistique | Apprend des poids sur les mots-clés | Section 1.2 |
| ③ Réseau de neurones (sans attention) | Apprend des combinaisons non-linéaires des mêmes mots-clés | nouveau ici |
| ④ Transformer pré-entraîné | Lit le texte brut, comprend le sens | nouveau ici |

Les méthodes ① ② ③ travaillent sur **les mêmes features binaires** (présence/absence de nos mots-clés). Seule ④ regarde directement le texte.

In [ ]:
from sklearn.neural_network import MLPClassifier

# ③ Réseau de neurones simple : MLP à 2 couches cachées
# Entraîné sur les mêmes features binaires que la régression logistique
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000, random_state=42)
mlp.fit(X_train, y_train)
print("Précision train MLP :", round(mlp.score(X_train, y_train), 2))
print("Précision test  MLP :", round(mlp.score(X_test, y_test), 2))

In [ ]:
# Préparer les features binaires des 4 phrases piégeuses
feats_pieges = pd.DataFrame([extraire_features(t) for t in phrases_pieges["texte"]])

# Lancer les 3 premières méthodes
resultats = phrases_pieges.copy()
resultats["① règles"]   = resultats["texte"].apply(predire_par_regles)
resultats["② log_reg"]  = modele.predict(feats_pieges)
resultats["③ mlp"]      = mlp.predict(feats_pieges)
resultats

**Observation.** Les méthodes ① ② ③ utilisent toutes les mêmes mots-clés en entrée. Comme aucun mot-clé n'apparaît dans les 4 phrases piégeuses, elles n'ont rien à se mettre sous la dent et **tombent dans le panneau**.

Augmenter la complexité du modèle (régression → réseau de neurones) ne sauve pas la mise : si l'entrée est vide, la sortie est aléatoire.

### 1.4.3 La méthode ④ : un transformer pré-entraîné

In [ ]:
# ④ Transformer pré-entraîné en mode "zero-shot"
# (zero-shot = pas d'entraînement spécifique sur nos partis ; le modèle utilise
#  uniquement la connaissance générale acquise lors de son pré-entraînement)
#
# from transformers import pipeline
# clf_transformer = pipeline(
#     "zero-shot-classification",
#     model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
# )
#
# def classer_transformer(texte):
#     return clf_transformer(texte, candidate_labels=["RN", "LFI", "LR", "EELV"])["labels"][0]
#
# resultats["④ transformer"] = resultats["texte"].apply(classer_transformer)
# resultats

**Pourquoi ça marche cette fois ?** Le transformer ne regarde pas nos mots-clés. Il lit le texte brut et y reconnaît **le sens** — il sait que « rémunération » et « salaire » sont proches, que « écosystème » et « écologie » sont proches.

Cette capacité à comprendre le sens **indépendamment des mots exacts**, c'est exactement ce qui a ouvert la voie à l'IA générative qu'on voit en 1.5.

### Hack Time

- Inventez une 5ᵉ phrase piégeuse pour un parti de votre choix (sans utiliser nos mots-clés).
- Lesquelles des 4 méthodes tombent en panne sur votre phrase ?
- Bonus : ajoutez 2-3 synonymes des mots-clés (par exemple `étranger` à côté de `immigration`) dans `MOTS_CLES` et relancez la section 1.2 + 1.4. À partir de quel point les méthodes ① ② ③ commencent-elles à rattraper le transformer ?

## 1.5 IA générative (2020 → aujourd'hui)

Le transformer de 1.4 « comprend » le texte. Mais il faut une dernière bascule pour qu'il **génère** du texte : on l'utilise dans une variante **décodeur** (le modèle ne voit que les mots à sa gauche et apprend à prédire le suivant).

C'est ce qui a donné ChatGPT, Claude, Mistral, Gemini : le modèle ne se contente plus de classer, il **génère sa réponse en langage naturel — et peut expliquer son choix**.

Magique ? Pas vraiment. Sous le capot, c'est toujours du transformer. Mais la combinaison **échelle massive + affinage par retours humains** rend le résultat saisissant.

> Note : on dialogue avec un grand modèle de langage via une **API**, c'est-à-dire un service en ligne auquel on envoie du texte et qui nous renvoie une réponse. Pour s'authentifier, on utilise une « clé d'API » (un long mot de passe technique) qu'on stocke dans un fichier `.env` à la racine du projet — exactement comme dans le notebook d'analyse des participants.

In [ ]:
# En séance : appel à un grand modèle de langage
# (la clé ANTHROPIC_API_KEY est lue depuis le fichier .env à la racine du repo)
#
# from anthropic import Anthropic
# client = Anthropic()
#
# msg = client.messages.create(
#     model="claude-haiku-4-5-20251001",
#     max_tokens=256,
#     messages=[{
#         "role": "user",
#         "content": "Classe cette déclaration parmi {RN, LFI, LR, EELV} "
#                    "et justifie en une phrase : "
#                    "'Augmentons les salaires et taxons les plus riches.'"
#     }],
# )
# print(msg.content[0].text)

### Hack Time

Une fois la cellule exécutée :
- Relancez 3 fois la même requête. Obtenez-vous toujours la même réponse ? (Indice : non.)
- Essayez de faire dire au modèle qu'une phrase RN est en fait du LFI. Qui gagne, vous ou le modèle ?
- Mesurez intuitivement : sur 10 phrases, combien le modèle classerait correctement ? Comparez à votre intuition pour les règles de 1.1 et à la régression logistique de 1.2.

## Récap module 1

| Époque | Approche | Forces | Limites |
|---|---|---|---|
| 1.1 IA symbolique | Règles à la main | Lisible, contrôlable | Tombe sur les synonymes |
| 1.2 ML statistique | Features à la main + régression apprise | Interprétable, des poids depuis les données | Limitée aux mots-clés choisis |
| 1.3 Deep Learning | Représentation apprise (réseau de neurones) | Capture des combinaisons non-linéaires | Toujours limité par les features d'entrée |
| 1.4 Transformer | Pré-entraînement massif + attention | Comprend le sens, gère les paraphrases | Boîte noire |
| 1.5 LLM génératif | Le transformer en mode décodeur | Génération libre + explication | Hallucinations, coût |

→ Direction le **module 2** : les concepts ML (features, split, overfitting, biais-variance) qui structurent **tout** ça — y compris ce qui se passe dans le ventre d'un LLM.